# Inventory and Zenodo archive verification

Create the deterministic raw inventory and compare path, size, and CRC32 against the published ZIP central directory.

Documentation authority: [Zenodo record](https://zenodo.org/records/8089820), [Scientific Data descriptor](https://www.nature.com/articles/s41597-023-02445-z), and the publisher documentation retained in `raw/`. Unknown semantics always force preserve-only behavior.

## Paths and safety guards

The enclosing `BCI Database/` is treated as the raw corpus. The project subtree is explicitly excluded from every raw traversal.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "notebooks").is_dir():
    raise RuntimeError("Run this notebook from bci_cleaning/ or bci_cleaning/notebooks/")

RAW_PATH = PROJECT_ROOT / "raw"
CLEANED_PATH = PROJECT_ROOT / "cleaned"
REPORTS_PATH = PROJECT_ROOT / "reports"
LOGS_PATH = PROJECT_ROOT / "logs"

if not RAW_PATH.is_symlink() or RAW_PATH.resolve(strict=True) != PROJECT_ROOT.parent.resolve(strict=True):
    raise RuntimeError("raw/ must remain a relative symlink to the enclosing BCI Database directory")

os.chdir(PROJECT_ROOT)
support_path = str(PROJECT_ROOT / "support")
if support_path not in sys.path:
    sys.path.insert(0, support_path)


## Phase implementation

In [2]:
"""Create a deterministic, hash-backed inventory of the untouched raw dataset."""

from __future__ import annotations

import sys
from collections import Counter

from bci_core import (
    ARCHIVE_MD5,
    INVENTORY_FIELDS,
    ZENODO_RECORD,
    ZENODO_URL,
    Issue,
    atomic_write_csv,
    atomic_write_text,
    common_parser,
    compare_archive,
    fetch_archive_manifest,
    inventory_rows,
    is_administrative_artifact,
    log_event,
    merge_issues,
    new_run_id,
    resolve_raw,
    write_structure_report,
)


ARCHIVE_FIELDS = ["archive_path", "size_bytes", "compressed_size_bytes", "crc32"]
COMPARISON_FIELDS = [
    "relative_path",
    "archive_path",
    "local_size_bytes",
    "archive_size_bytes",
    "local_crc32",
    "archive_crc32",
    "status",
]


In [3]:
def main() -> int:
    parser = common_parser(__doc__ or "Inventory raw dataset")
    parser.add_argument("--archive-url", default=ZENODO_URL)
    parser.add_argument(
        "--skip-archive-check",
        action="store_true",
        help="Create a local inventory only. Cleaning remains blocked until archive comparison passes.",
    )
    args = parser.parse_args()
    raw = resolve_raw(args.raw)
    args.reports.mkdir(parents=True, exist_ok=True)
    args.logs.mkdir(parents=True, exist_ok=True)
    run_id = new_run_id()
    issues: list[Issue] = []
    log_event(args.logs, run_id, "inventory", "start", raw=str(raw))

    try:
        rows = inventory_rows(raw, args.logs, run_id)
    except Exception as exc:
        log_event(args.logs, run_id, "inventory", "failed", error=type(exc).__name__)
        raise

    atomic_write_csv(args.reports / "dataset_inventory.csv", rows, INVENTORY_FIELDS)
    write_structure_report(raw, rows, args.reports / "dataset_structure.md")

    for row in rows:
        relative = row["relative_path"]
        if row["readable"] != "true":
            issues.append(
                Issue(
                    "error", "inventory", relative, "", "readability", "unreadable",
                    "Every raw file must be readable", ZENODO_RECORD,
                    "Restore file permissions or re-extract the archive.",
                )
            )
        if int(row["size_bytes"]) == 0:
            issues.append(
                Issue(
                    "warning", "inventory", relative, "", "empty_file", "0 bytes",
                    "Empty files require documentation before interpretation", ZENODO_RECORD,
                    "Preserve the raw file and review its documented role.",
                )
            )

    archive_ok = False
    if not args.skip_archive_check:
        try:
            archive_size, archive_rows = fetch_archive_manifest(args.archive_url)
            atomic_write_csv(args.reports / "archive_manifest.csv", archive_rows, ARCHIVE_FIELDS)
            comparison = compare_archive(raw.name, rows, archive_rows)
            for result in comparison:
                relative = result["relative_path"]
                local_path = raw / relative
                if (
                    result["status"] == "extra_local"
                    and local_path.is_file()
                    and is_administrative_artifact(relative, local_path)
                ):
                    result["status"] = "administrative_extra"
            atomic_write_csv(args.reports / "archive_comparison.csv", comparison, COMPARISON_FIELDS)
            mismatches = [
                row for row in comparison if row["status"] not in {"match", "administrative_extra"}
            ]
            archive_ok = not mismatches
            for administrative in (
                row for row in comparison if row["status"] == "administrative_extra"
            ):
                issues.append(
                    Issue(
                        "info", "archive", administrative["relative_path"], "",
                        "zenodo_member_integrity", "administrative_extra",
                        "Non-research OS metadata may exist locally but is not an archive member",
                        ZENODO_RECORD,
                        "Retain in raw, exclude from cleaned, and record in the cleaning manifest.",
                    )
                )
            for mismatch in mismatches:
                issues.append(
                    Issue(
                        "fatal",
                        "archive",
                        mismatch["relative_path"],
                        "",
                        "zenodo_member_integrity",
                        mismatch["status"],
                        "Local path, size, and CRC32 must match the Zenodo ZIP member",
                        ZENODO_RECORD,
                        "Re-extract or restore the exact published archive member before cleaning.",
                    )
                )
            administrative_count = sum(
                row["status"] == "administrative_extra" for row in comparison
            )
            metadata = [
                "# Zenodo Archive Verification",
                "",
                f"- Source: {args.archive_url}",
                f"- Zenodo record: {ZENODO_RECORD}",
                f"- Published archive MD5: `{ARCHIVE_MD5}`",
                f"- Remote archive size: {archive_size:,} bytes",
                f"- File members: {len(archive_rows):,}",
                f"- Local files: {len(rows):,}",
                f"- Non-research administrative extras: {administrative_count:,}",
                f"- Result: {'PASS' if archive_ok else 'FAIL'}",
                "",
                "The archive itself is not downloaded. Verification uses its central-directory sizes and CRC32 values.",
                "",
            ]
            atomic_write_text(args.reports / "archive_verification.md", "\n".join(metadata))
        except Exception as exc:
            issues.append(
                Issue(
                    "fatal", "archive", "", "", "zenodo_manifest_fetch",
                    f"{type(exc).__name__}: {exc}",
                    "Archive membership must be verified before cleaning", ZENODO_RECORD,
                    "Restore network access and rerun inventory; do not clean from an unverified extraction.",
                )
            )
    else:
        issues.append(
            Issue(
                "warning", "archive", "", "", "zenodo_manifest_fetch", "skipped",
                "Archive membership must be verified before cleaning", ZENODO_RECORD,
                "Rerun without --skip-archive-check before cleaning.",
            )
        )

    merge_issues(args.reports / "validation_issues.csv", issues, {"inventory", "archive"})
    counts = Counter((row["extension"] or "[no extension]") for row in rows)
    log_event(
        args.logs,
        run_id,
        "inventory",
        "complete",
        files=len(rows),
        size_bytes=sum(int(row["size_bytes"]) for row in rows),
        extension_counts=dict(sorted(counts.items())),
        archive_verified=archive_ok,
        issue_count=len(issues),
    )
    print(f"Inventory complete: {len(rows)} files")
    print(f"Archive verification: {'PASS' if archive_ok else 'NOT PASSED'}")
    return 0 if not any(issue.severity in {"fatal", "error"} for issue in issues) else 2


## Execute phase

In [4]:
_arguments = [
    "01_inventory.ipynb",
    "--raw", str(RAW_PATH),
    "--cleaned", str(CLEANED_PATH),
    "--reports", str(REPORTS_PATH),
    "--logs", str(LOGS_PATH),
]

ARCHIVE_URL_OVERRIDE = None
SKIP_ARCHIVE_CHECK = False
if ARCHIVE_URL_OVERRIDE:
    _arguments.extend(["--archive-url", ARCHIVE_URL_OVERRIDE])
if SKIP_ARCHIVE_CHECK:
    _arguments.append("--skip-archive-check")

_original_argv = sys.argv[:]
try:
    sys.argv = _arguments
    _exit_code = main()
finally:
    sys.argv = _original_argv

if _exit_code != 0:
    raise RuntimeError(f"Phase returned nonzero status {_exit_code}; inspect validation_issues.csv")


Inventory complete: 1073 files
Archive verification: PASS


## Privacy-safe report check

In [5]:
import csv

report_path = REPORTS_PATH / "dataset_inventory.csv"
with report_path.open("r", encoding="utf-8", newline="") as handle:
    report_rows = list(csv.DictReader(handle))

{
    "report": str(report_path.relative_to(PROJECT_ROOT)),
    "rows": len(report_rows),
}


{'report': 'reports/dataset_inventory.csv', 'rows': 1073}